# 18. Scaled Helpdesk Benchmark: Vector RAG vs a Graph Agent on ~190 Template Questions (Aura Graph)

**Presentation map:** Slide 32 (vector RAG is the baseline), Slide 36 (graphs help on reasoning, not on lookup),
Slide 38 (decision rule by question type), Slide 43 (exposing a graph to an agent as tools).

Notebook 1 compared vector RAG with a graph agent on **22 hand-picked questions** (5 lookup, 11 verified-fix, 6
relational) — enough to show the pattern, too few for confidence intervals narrower than ±25 points. This notebook
keeps the same graph, the same two systems and the same scoring, and multiplies the questions with **templates whose
gold answers are computed from the graph**:

| Family | How the questions are made | n |
|---|---|---|
| **A. Single-hop lookup** | each of the 10 KB articles, asked for in 3 different wordings (LLM paraphrases) | 30 |
| **B. Verified fix** | each of the 11 closed tickets with an accepted solution, re-told as a new report in 4 wordings | 44 |
| **C. Relational / aggregate** | 9 templates filled with every user, category, priority, role or ticket — counts, filters, joins | ≈115 |

**Systems (notebook 1's, unchanged):** (a) vector RAG over 90 chunks with metadata headers, top-8; (b) a
tool-calling graph agent with `semantic_search`, `find_nodes`, `count_nodes`, `get_node`, `get_verified_solution`
(≤ 10 steps). **Scoring:** families A and C by matching the gold ids/names in the answer (and none of the wrong ones);
family B by a strict LLM judge (same resolution as the verified fix).

**Model:** `gpt-6-luna`, no reasoning, via the OpenAI Batch API (`smash_batch.py`); the agent runs in lock-step, one
Batch job per step. Cached in `data/cache_18/`.

In [1]:
import json, os, re, hashlib
from collections import Counter, defaultdict
import numpy as np
import networkx as nx
from pydantic import BaseModel
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()
TEST = MODE == "test"
llm = LLM("data/cache_18" + ("_test" if TEST else ""))

data = json.load(open("data/aura_helpdesk.json"))
G = nx.MultiDiGraph()
for key, lab, idf in (("categories", "Category", "categoryId"), ("priorities", "Priority", "priorityId"),
                      ("statuses", "Status", "statusId"), ("users", "User", "userId"), ("tickets", "Ticket", "ticketId"),
                      ("comments", "Comment", "commentId"), ("documentation", "Documentation", "articleId")):
    for x in data[key]:
        G.add_node(x[idf], label=lab, **x)
for rel in data["relationships"]:
    G.add_edge(rel["from"], rel["to"], key=rel["type"], type=rel["type"])
for n, d in G.nodes(data=True):
    if "label" not in d:
        d.update(label="Category", name="(undefined category)")

def out_nbrs(n, r): return [v for _, v, d in G.out_edges(n, data=True) if d["type"] == r]
def in_nbrs(n, r):  return [u for u, _, d in G.in_edges(n, data=True) if d["type"] == r]
def user_name(u):   return f"{G.nodes[u]['firstName']} {G.nodes[u]['lastName']}" if u in G else None
def ticket_status(t): return G.nodes[out_nbrs(t, "HAS_STATUS")[0]]
def ticket_closed(t): return ticket_status(t)["isClosed"] == "true"
def nodes(label): return sorted(n for n, d in G.nodes(data=True) if d["label"] == label)
print(f"Aura graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges |",
      dict(Counter(nx.get_node_attributes(G, 'label').values())))

Aura graph: 122 nodes, 246 edges | {'Category': 10, 'Priority': 5, 'Status': 7, 'User': 10, 'Ticket': 23, 'Comment': 57, 'Documentation': 10}


## The vector index (notebook 1's 90 chunks with metadata headers)

In [2]:
chunks = []
for t in data["tickets"]:
    tid, st = t["ticketId"], ticket_status(t["ticketId"])
    cat = G.nodes[t["categoryId"]]["name"] if t["categoryId"] in G else "unknown"
    chunks.append({"id": tid, "label": "Ticket", "text":
        f"[Ticket {tid} | category: {cat} | status: {st['name']} | submitted by {user_name(t['createdByUserId'])} | "
        f"assigned to {user_name(t['assignedToUserId'])}]\n{t['title']}. {t['description']}\n"
        f"Resolution summary: {t['resolutionSummary'] or 'none'}"})
for c in data["comments"]:
    a = G.nodes[c["authorUserId"]]
    chunks.append({"id": c["commentId"], "label": "Comment", "text":
        f"[Comment {c['commentId']} on ticket {c['ticketId']} | by {user_name(c['authorUserId'])} ({a['role']}) | "
        f"accepted solution: {'yes' if c['isSolution'] == 'True' else 'no'}]\n{c['content']}"})
for d in data["documentation"]:
    chunks.append({"id": d["articleId"], "label": "Documentation", "text": f"[KB article {d['articleId']}]\n{d['title']}: {d['summary']}"})
CV = llm.embed("chunks", [c["text"] for c in chunks])

QV_PATH = f"{llm.cache}/query_vecs.json"
QV = json.load(open(QV_PATH)) if os.path.exists(QV_PATH) else {}
def qvec(texts):
    # Query embeddings, cached as JSON so that replay needs no API call.
    miss = [t for t in dict.fromkeys(texts) if t not in QV]
    if miss:
        if MODE == "replay":
            raise RuntimeError("query embedding not cached - run with SMASH_MODE=test or batch")
        r = llm.client.embeddings.create(model="text-embedding-3-small", input=miss)
        for t, e in zip(miss, r.data):
            v = np.array(e.embedding); QV[t] = list(np.round(v / np.linalg.norm(v), 6))
        json.dump(QV, open(QV_PATH, "w"))
    return np.array([QV[t] for t in texts])

def vector_search(query, k=8, label=None):
    sims = CV @ qvec([query])[0]
    order = [i for i in np.argsort(-sims) if label is None or chunks[i]["label"] == label]
    return [(chunks[i], float(sims[i])) for i in order[:k]]
print(f"{len(chunks)} chunks")

90 chunks


## Questions with gold answers computed from the graph

In [3]:
alias = lambda u: [u, user_name(u).lower()]
NONE = ["none", "zero", "no tickets", "no open", "not any", "0"]
cnt = lambda n: [str(n)] + (NONE if n == 0 else [])      # a count of 0 may be written as "none"
tickets, users, docs = nodes("Ticket"), nodes("User"), nodes("Documentation")
cats = [c for c in nodes("Category") if G.nodes[c]["name"] != "(undefined category)"]
def fix_of(t):
    if not ticket_closed(t):
        return None
    for c in out_nbrs(t, "CONTAINS"):
        if G.nodes[c]["isSolution"] == "True":
            a = in_nbrs(c, "POSTS")[0]
            return {"ticket": t, "comment": c, "text": G.nodes[c]["content"], "author": user_name(a), "role": G.nodes[a]["role"]}
fix_tickets = [t for t in tickets if fix_of(t)]

class Variants(BaseModel):
    items: list[str]
PARA = {f"doc|{d}": LLM.body([{"role": "user", "content": "Write 3 different questions a user might ask whose answer is "
                               f"this help article (do not quote its title): {G.nodes[d]['title']}: {G.nodes[d]['summary']}"}],
                             max_tokens=300, schema=Variants) for d in docs}
PARA.update({f"fix|{t}": LLM.body([{"role": "user", "content": "Rewrite this support ticket as 4 different NEW reports from "
                                    "different employees (1-2 sentences each, same problem, different wording, no ids):\n"
                                    f"{G.nodes[t]['title']}. {G.nodes[t]['description']}"}], max_tokens=500, schema=Variants)
             for t in fix_tickets})
pv = llm.run("paraphrases", PARA, parse=Variants)
Q = []
for d in docs:
    for v in (pv[f"doc|{d}"]["parsed"] or {"items": []})["items"][:3]:
        Q.append({"family": "A. lookup", "q": v, "gold": {"all": [[d, G.nodes[d]["title"].lower()]]}})
for t in fix_tickets:
    fx = fix_of(t)
    for v in (pv[f"fix|{t}"]["parsed"] or {"items": []})["items"][:4]:
        Q.append({"family": "B. verified fix", "q": f"New ticket: {v} What should the user do?",
                  "gold": {"comment": fx["comment"], "fix_text": fx["text"]}})

C = []
for u in users:
    assigned = [t for t in tickets if in_nbrs(t, "ASSIGNED_TO") == [u]]
    if assigned:
        C.append((f"How many tickets are assigned to {user_name(u)}?", {"all": [cnt(len(assigned))]}))
        C.append((f"Which tickets assigned to {user_name(u)} are not yet closed?",
                  {"all": [[t, G.nodes[t]["title"].lower()] for t in assigned if not ticket_closed(t)] or [NONE],
                   "none": [[t] for t in assigned if ticket_closed(t)]}))
    sub = [t for t in tickets if in_nbrs(t, "SUBMITS") == [u]]
    if sub:
        C.append((f"How many of the tickets submitted by {user_name(u)} are still not closed?",
                  {"all": [cnt(sum(not ticket_closed(t) for t in sub))]}))
    ncom = len([c for c in nodes("Comment") if in_nbrs(c, "POSTS") == [u]])
    C.append((f"How many comments has {user_name(u)} posted?", {"all": [cnt(ncom)]}))
for c in cats:
    in_cat = [t for t in tickets if c in out_nbrs(t, "HAS_CATEGORY")]
    if in_cat:
        C.append((f"How many tickets are in the category '{G.nodes[c]['name']}', and how many of them are closed?",
                  {"all": [cnt(len(in_cat)), cnt(sum(ticket_closed(t) for t in in_cat))]}))
for p in nodes("Priority"):
    n = len([t for t in tickets if p in out_nbrs(t, "HAS_PRIORITY")])
    C.append((f"How many tickets have the priority '{G.nodes[p]['name']}'?", {"all": [cnt(n)]}))
for t in fix_tickets:
    fx = fix_of(t)
    C.append((f"Who wrote the accepted solution for the ticket '{G.nodes[t]['title']}', and what is their role?",
              {"all": [[fx["author"].lower()], [fx["role"].lower()]]}))
for t in tickets:
    refs = sorted({d for c in out_nbrs(t, "CONTAINS") for d in in_nbrs(c, "REFERENCED_BY")})
    if refs:
        C.append((f"Which KB articles are referenced in the comments of the ticket '{G.nodes[t]['title']}'?",
                  {"all": [[d, G.nodes[d]["title"].lower()] for d in refs], "none": [[d] for d in docs if d not in refs]}))
    C.append((f"What is the status of the ticket '{G.nodes[t]['title']}', and who is it assigned to?",
              {"all": [[ticket_status(t)["name"].lower()], alias(in_nbrs(t, "ASSIGNED_TO")[0])]}))
for role in sorted({G.nodes[u]["role"] for u in users}):
    solvers = sorted({in_nbrs(c, "POSTS")[0] for c in nodes("Comment") if G.nodes[c]["isSolution"] == "True"
                      and G.nodes[in_nbrs(c, "POSTS")[0]]["role"] == role})
    if solvers:
        C.append((f"Which users with the role '{role}' have written at least one accepted solution?",
                  {"all": [alias(u) for u in solvers],
                   "none": [alias(u) for u in users if G.nodes[u]["role"] == role and u not in solvers]}))
Q += [{"family": "C. relational", "q": q, "gold": g} for q, g in C]
for i, q in enumerate(Q):
    q["id"] = f"q{i:03d}"
if TEST:
    Q = [q for fam in ("A. lookup", "B. verified fix", "C. relational") for q in [x for x in Q if x["family"] == fam][:5]]
print(f"{len(Q)} questions |", dict(Counter(q["family"] for q in Q)))
for q in [x for x in Q if x["family"] == "C. relational"][:4]:
    print(f"  - {q['q']}\n      gold: {q['gold']['all'][:3]}")

156 questions | {'A. lookup': 30, 'B. verified fix': 44, 'C. relational': 82}
  - How many of the tickets submitted by Marie Dubois are still not closed?
      gold: [['0', 'none', 'zero', 'no tickets', 'no open', 'not any', '0']]
  - How many comments has Marie Dubois posted?
      gold: [['3']]
  - How many tickets are assigned to Jean Martin?
      gold: [['4']]
  - Which tickets assigned to Jean Martin are not yet closed?
      gold: [['tkt_008', 'mobile access to aura not working']]


## (a) Vector RAG — one Batch job

In [4]:
RULES = ("Answer using only the provided context. Cite the ids (tkt_..., cmt_..., doc_...) that support your answer. "
         "If the context is insufficient, say so.")
vec_ctx = {q["id"]: vector_search(q["q"]) for q in Q}
VEC = llm.run("vector", {q["id"]: LLM.body([{"role": "system", "content": "You are the Aura IT helpdesk assistant. " + RULES},
                                            {"role": "user", "content": f"Question: {q['q']}\n\nContext:\n" +
                                             "\n\n".join(c['text'] for c, _ in vec_ctx[q['id']])}], max_tokens=800) for q in Q})

## (b) The graph agent — all questions advance one tool step per Batch job

In [5]:
def node_view(n):
    d = G.nodes[n]; lab = d["label"]
    if lab == "Ticket":
        st = ticket_status(n)
        return {"id": n, "label": lab, "title": d["title"], "status": st["name"], "isClosed": st["isClosed"],
                "category": G.nodes[d["categoryId"]]["name"] if d["categoryId"] in G else None,
                "priority": G.nodes[out_nbrs(n, "HAS_PRIORITY")[0]]["name"] if out_nbrs(n, "HAS_PRIORITY") else None,
                "assignee": user_name(d["assignedToUserId"]), "submitter": user_name(d["createdByUserId"])}
    if lab == "Comment":
        a = in_nbrs(n, "POSTS")[0]
        return {"id": n, "label": lab, "ticket": d["ticketId"], "isSolution": d["isSolution"], "author": user_name(a),
                "author_role": G.nodes[a]["role"], "text": d["content"][:160], "references_kb": in_nbrs(n, "REFERENCED_BY")}
    if lab == "User":
        return {"id": n, "label": lab, "name": user_name(n), "role": d["role"]}
    return {"id": n, "label": lab, **{k: v for k, v in d.items() if k in ("name", "title", "summary", "isClosed")}}

def semantic_search(query, label=None, k=5):
    return json.dumps([{"id": c["id"], "score": round(s, 3), "text": c["text"][:300]} for c, s in vector_search(query, k, label)])
def find_nodes(label, filters=None):
    rows = [node_view(n) for n in nodes(label)]
    for key, val in (filters or {}).items():
        rows = [r for r in rows if str(r.get(key, "")).lower() == str(val).lower()]
    return json.dumps(rows)
def count_nodes(label, group_by, filters=None):
    groups = defaultdict(list)
    for r in json.loads(find_nodes(label, filters)):
        groups[str(r.get(group_by))].append(r["id"])
    return json.dumps({k: {"count": len(v), "ids": v} for k, v in sorted(groups.items(), key=lambda kv: -len(kv[1]))})
def get_node(node_id):
    if node_id not in G:
        return json.dumps({"error": f"no node {node_id}"})
    edges = [{"rel": d["type"], "dir": "out", "node": node_view(v)} for _, v, d in G.out_edges(node_id, data=True)]
    edges += [{"rel": d["type"], "dir": "in", "node": node_view(u)} for u, _, d in G.in_edges(node_id, data=True)]
    return json.dumps({"node": node_view(node_id), "edges": edges})
def get_verified_solution(ticket_id):
    return json.dumps(fix_of(ticket_id) if ticket_id in G and G.nodes[ticket_id]["label"] == "Ticket" and fix_of(ticket_id)
                      else {"ticket": ticket_id, "verified_solution": None})

IMPL = {"semantic_search": semantic_search, "find_nodes": find_nodes, "count_nodes": count_nodes, "get_node": get_node,
        "get_verified_solution": get_verified_solution}
S_ = lambda **p: {"type": "object", "properties": p, "required": [next(iter(p))]}
TOOLS = [
    {"type": "function", "function": {"name": "semantic_search", "description": "Vector search over tickets, comments and KB articles. Optional label: Ticket | Comment | Documentation.",
     "parameters": S_(query={"type": "string"}, label={"type": "string"}, k={"type": "integer"})}},
    {"type": "function", "function": {"name": "find_nodes", "description": "Return all nodes of a label (Ticket, Comment, User, Documentation, Status, Category, Priority) as joined rows, optionally filtered by exact field values, e.g. {'isClosed': 'false', 'category': 'Aura Access'} for Tickets or {'isSolution': 'True'} for Comments.",
     "parameters": S_(label={"type": "string"}, filters={"type": "object", "additionalProperties": {"type": "string"}})}},
    {"type": "function", "function": {"name": "count_nodes", "description": "Count nodes of a label grouped by one field (e.g. Comment grouped by 'author' with {'isSolution': 'True'}; Ticket grouped by 'assignee' with {'isClosed': 'false'}). Returns count and ids per group. Use this for every 'how many / the most' question.",
     "parameters": S_(label={"type": "string"}, group_by={"type": "string"}, filters={"type": "object", "additionalProperties": {"type": "string"}})}},
    {"type": "function", "function": {"name": "get_node", "description": "A node with all its incoming/outgoing edges.", "parameters": S_(node_id={"type": "string"})}},
    {"type": "function", "function": {"name": "get_verified_solution", "description": "For a ticket: if it is closed, the comment marked as the accepted solution, with its author.", "parameters": S_(ticket_id={"type": "string"})}},
]
AGENT_SYS = ("You are the Aura IT helpdesk assistant with tools over the helpdesk knowledge graph. For a new problem "
             "report, find the most similar historical tickets (semantic_search with label='Ticket') and check their "
             "verified solutions. Let count_nodes do any counting. " + RULES)

state = {q["id"]: {"msgs": [{"role": "system", "content": AGENT_SYS}, {"role": "user", "content": q["q"]}],
                   "answer": None, "seen": [], "steps": 0} for q in Q}
for step in range(10):
    active = [k for k, s in state.items() if s["answer"] is None]
    if not active:
        break
    res = llm.run(f"agent_step{step + 1}", {k: LLM.body(state[k]["msgs"], max_tokens=800, tools=TOOLS) for k in active})
    for k in active:
        r, s = res[k], state[k]
        s["steps"] += 1
        if not r["tool_calls"]:
            s["answer"] = r["text"]; continue
        s["msgs"].append({"role": "assistant", "content": r["text"] or None,
                          "tool_calls": [{"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["arguments"]}}
                                         for t in r["tool_calls"]]})
        for t in r["tool_calls"]:
            try:
                out = IMPL[t["name"]](**json.loads(t["arguments"]))
            except Exception as e:
                out = json.dumps({"error": f"{type(e).__name__}: {e}"})
            s["seen"] += re.findall(r"(?:tkt|cmt|doc)_\d{3}", out)
            s["msgs"].append({"role": "tool", "tool_call_id": t["id"], "content": out})
    print(f"step {step + 1}: {len(active)} agents active")
for s in state.values():
    if s["answer"] is None:
        s["answer"] = "(no answer within step budget)"
print(f"Average tool steps per question: {np.mean([s['steps'] for s in state.values()]):.1f}")

step 1: 156 agents active
step 2: 156 agents active
step 3: 104 agents active
step 4: 45 agents active
step 5: 24 agents active
step 6: 16 agents active


step 7: 13 agents active
step 8: 10 agents active
step 9: 9 agents active
step 10: 7 agents active
Average tool steps per question: 3.5


## Scoring

In [6]:
class Verdict(BaseModel):
    same_fix: bool
B = [q for q in Q if q["family"] == "B. verified fix"]
jr = llm.run("judge", {f"{m}|{q['id']}": LLM.body([
    {"role": "system", "content": "You grade helpdesk answers strictly. same_fix=true only if the candidate's main "
     "recommendation is the same resolution as the reference verified fix (not merely related advice)."},
    {"role": "user", "content": f"Reference verified fix:\n{q['gold']['fix_text']}\n\nCandidate answer:\n{ans}"}], max_tokens=20, schema=Verdict)
    for q in B for m, ans in (("vector", VEC[q["id"]]["text"]), ("graph", state[q["id"]]["answer"]))}, parse=Verdict)

def has(ans, aliases): return any(re.search(rf"(?<![\w.]){re.escape(a)}(?![\w])", ans.lower()) for a in aliases)
def correct(q, m):
    ans = VEC[q["id"]]["text"] if m == "vector" else state[q["id"]]["answer"]
    g = q["gold"]
    if "fix_text" in g:
        return bool((jr[f"{m}|{q['id']}"]["parsed"] or {}).get("same_fix"))
    return all(has(ans, al) for al in g["all"]) and not any(has(ans, al) for al in g.get("none", []))

OK = {(q["id"], m): correct(q, m) for q in Q for m in ("vector", "graph")}
FAMS = ["A. lookup", "B. verified fix", "C. relational"]
RES = {}
print(f"{'family':<18}{'n':>4}{'vector [95% CI]':>22}{'graph agent [95% CI]':>24}{'graph - vector [95% CI]':>28}")
for f in FAMS + ["ALL"]:
    qs = [q for q in Q if f == "ALL" or q["family"] == f]
    row = {}
    for m in ("vector", "graph"):
        k = sum(OK[(q["id"], m)] for q in qs); lo, hi = wilson(k, len(qs)); row[m] = (k / len(qs), lo, hi)
    d = paired_boot([OK[(q["id"], "vector")] for q in qs], [OK[(q["id"], "graph")] for q in qs])
    RES[f] = (len(qs), row, d)
    fmt = lambda t: f"{t[0]:.0%} [{t[1]:.0%}-{t[2]:.0%}]"
    print(f"{f:<18}{len(qs):>4}{fmt(row['vector']):>22}{fmt(row['graph']):>24}{f'{d[0]:+.0%} [{d[1]:+.0%}, {d[2]:+.0%}]':>28}")
vb = sum(q["gold"]["comment"] in [c["id"] for c, _ in vec_ctx[q["id"]]] for q in B)
gb = sum(q["gold"]["comment"] in state[q["id"]]["seen"] for q in B)
print(f"\nVerified-fix comment present in context: vector {vb}/{len(B)}, graph {gb}/{len(B)}")

family               n       vector [95% CI]    graph agent [95% CI]     graph - vector [95% CI]


A. lookup           30         97% [83%-99%]           47% [30%-64%]           -50% [-67%, -33%]
B. verified fix     44         73% [58%-84%]           89% [76%-95%]            +16% [+2%, +30%]
C. relational       82         46% [36%-57%]           98% [92%-99%]           +51% [+40%, +62%]
ALL                156         63% [56%-71%]           85% [79%-90%]           +22% [+12%, +31%]

Verified-fix comment present in context: vector 19/44, graph 44/44


## Cost

In [7]:
LEDGER = llm.print_ledger()

stage                           calls     input tok   output tok      USD  via Batch
agent_step1                       156        77,875        5,357     0.01       100%
agent_step10                        7        27,841          420     0.00       100%
agent_step2                       156       140,470        4,470     0.01       100%
agent_step3                       104       123,433        4,151     0.01       100%
agent_step4                        45        74,725        2,013     0.00       100%
agent_step5                        24        47,951        1,017     0.00       100%
agent_step6                        16        41,105          547     0.00       100%
agent_step7                        13        38,944          607     0.00       100%
agent_step8                        10        32,813          363     0.00       100%
agent_step9                         9        34,604          542     0.00       100%
judge                              88        15,950        1,144 

In [8]:
print("=" * 92)
print(f"RESULTS SUMMARY — Notebook 18: Aura helpdesk, {len(Q)} template questions, gpt-6-luna (notebook 1 had 22)")
print("=" * 92)
for f in FAMS + ["ALL"]:
    n, row, d = RES[f]
    verdict = "graph better" if d[1] > 0 else "vector better" if d[2] < 0 else "no resolvable difference"
    print(f"{f:<18} n={n:<4} vector {row['vector'][0]:.0%} [{row['vector'][1]:.0%}-{row['vector'][2]:.0%}] | "
          f"graph {row['graph'][0]:.0%} [{row['graph'][1]:.0%}-{row['graph'][2]:.0%}] | {verdict}")
print("\nLecture message (Slides 32/36/38): the same decision rule as notebook 1, now with intervals narrow enough to"
      "\nread by family — compare the verdict per family rather than the overall number.")

RESULTS SUMMARY — Notebook 18: Aura helpdesk, 156 template questions, gpt-6-luna (notebook 1 had 22)
A. lookup          n=30   vector 97% [83%-99%] | graph 47% [30%-64%] | vector better
B. verified fix    n=44   vector 73% [58%-84%] | graph 89% [76%-95%] | graph better
C. relational      n=82   vector 46% [36%-57%] | graph 98% [92%-99%] | graph better
ALL                n=156  vector 63% [56%-71%] | graph 85% [79%-90%] | graph better

Lecture message (Slides 32/36/38): the same decision rule as notebook 1, now with intervals narrow enough to
read by family — compare the verdict per family rather than the overall number.


## Added after review: questions written without the graph

The questions above come from templates whose gold answers are computed from the same graph the agent queries, and
the templates ask exactly what the agent's tools (`count_nodes`, `find_nodes`) can compute. As a check that is
independent of the graph, an LLM now writes the questions **from the 90 text chunks only** (the vector index; no
graph, schema or tool list), in the three families, with its own gold answer and supporting chunk ids. A second call
that reads all 90 chunks answers every question; a question is kept only if that answer agrees with the writer's gold
(LLM equivalence check). Both systems then run unchanged on the kept questions and one strict LLM judge scores every
answer against the gold answer.

These few hundred requests are sent synchronously (standard tier, twice the Batch price) because the agent's lock-step Batch jobs queue for hours.

In [9]:
import random
if MODE == "batch":
    llm.mode = "test"   # ~1,000 small lock-step requests: the standard tier finishes in minutes, a Batch job per step can take hours
ALL_CHUNKS = "\n\n".join(c["text"] for c in chunks)
class WQ(BaseModel):
    question: str
    answer: str
    support_ids: list[str]
class WQs(BaseModel):
    items: list[WQ]
FAM_SPEC = {
    "A. lookup": ("lookup questions whose answer is stated in ONE chunk (a KB article, a ticket's status, assignee, "
                  "category or resolution, a comment's content)", 3),
    "B. verified fix": ("problem reports: describe, as a NEW employee would in 1-2 sentences, a problem that a past ticket "
                        "had, then ask what to do; the answer is the fix that resolved that ticket (prefer comments "
                        "marked as accepted solution)", 3),
    "C. relational": ("questions that need SEVERAL chunks to answer: counts, lists, comparisons, who/what across tickets, "
                      "users, comments and articles - choose yourself what a helpdesk manager would want to know", 6),
}
WRITER_SYS = ("You are a helpdesk team lead writing an evaluation set for a helpdesk assistant. You see the helpdesk's "
              "records as text. Write questions that staff and managers would really ask, with a short, checkable "
              "gold answer that is fully determined by the records. Do not mention record ids (tkt_..., cmt_..., "
              "doc_...) in the questions; use titles, names and descriptions instead.")
rng = random.Random(7)
WRITE = {}
for fam, (spec, ncalls) in FAM_SPEC.items():
    for j in range(ncalls):
        focus = rng.sample(range(len(chunks)), 15)
        WRITE[f"{fam}|{j}"] = LLM.body([{"role": "system", "content": WRITER_SYS}, {"role": "user", "content":
            f"All helpdesk records:\n\n{ALL_CHUNKS}\n\n---\nWrite 10 {spec}. Centre the questions on these records "
            f"(other records may be needed too): {', '.join(chunks[i]['id'] for i in focus)}. For each question give "
            "the gold answer (short; exact numbers and names) and the ids of all records that support it."}],
            max_tokens=3000, schema=WQs)
wr = llm.run("indep_writer", WRITE, parse=WQs)
IQ, seen_q = [], set()
for k, r in wr.items():
    fam = k.split("|")[0]
    for it in (r["parsed"] or {"items": []})["items"]:
        key = re.sub(r"\W+", " ", it["question"].lower()).strip()
        if key not in seen_q:
            seen_q.add(key)
            IQ.append({"family": fam, "q": it["question"], "gold": it["answer"], "support": it["support_ids"]})
for i, q in enumerate(IQ):
    q["id"] = f"iq{i:03d}"
print(f"{len(IQ)} written questions |", dict(Counter(q["family"] for q in IQ)))

[indep_writer] 12 new requests -> test (0 cached)


119 written questions | {'A. lookup': 29, 'B. verified fix': 30, 'C. relational': 60}


In [10]:
class Ans(BaseModel):
    answer: str
class Same(BaseModel):
    same: bool
val = llm.run("indep_validate", {q["id"]: LLM.body([{"role": "user", "content":
    f"Helpdesk records:\n\n{ALL_CHUNKS}\n\n---\nAnswer from these records only, briefly (exact numbers and names).\n"
    f"Question: {q['q']}"}], max_tokens=400, schema=Ans) for q in IQ}, parse=Ans)
EQ_SYS = ("Decide whether two answers to the same question state the same facts (same numbers, same names/items, same "
          "fix). Differences in wording, order or extra harmless detail do not matter; a missing or different item does.")
eqv = llm.run("indep_equiv", {q["id"]: LLM.body([{"role": "system", "content": EQ_SYS}, {"role": "user", "content":
    f"Question: {q['q']}\nAnswer A: {q['gold']}\nAnswer B: {(val[q['id']]['parsed'] or {}).get('answer', '')}"}],
    max_tokens=20, schema=Same) for q in IQ}, parse=Same)
KQ = [q for q in IQ if (eqv[q["id"]]["parsed"] or {}).get("same")]
print("kept after independent validation:", {f: f"{sum(q['family'] == f for q in KQ)}/{sum(q['family'] == f for q in IQ)}"
                                             for f in FAMS}, f"| total {len(KQ)}/{len(IQ)}")
for f in FAMS:
    for q in [x for x in KQ if x["family"] == f][:3]:
        print(f"  [{f}] {q['q']}\n      gold: {q['gold']}")

[indep_validate] 119 new requests -> test (0 cached)


[indep_equiv] 119 new requests -> test (0 cached)


kept after independent validation: {'A. lookup': '29/29', 'B. verified fix': '28/30', 'C. relational': '58/60'} | total 115/119
  [A. lookup] How many DELMIA team members did Carlos say would attend the training?
      gold: 8 team members.
  [A. lookup] What did Aura do incorrectly in response to Lisa’s holiday-related question?
      gold: It gave an irrelevant, funny response from the chitchat agent.
  [A. lookup] What did Jean provide Thomas to help him find Aura documentation?
      gold: Direct links to the Aura documentation, also added to the intranet quick links.
  [B. verified fix] I’m new to Aura and it works on my computer, but the page won’t load in either Safari or Chrome on my iPhone. What should I do?
      gold: The ticket did not record a fix; support was still investigating the mobile-browser issue.
  [B. verified fix] I’m new to Aura and want to train my DELMIA team to use it for manufacturing questions. What training was arranged?
      gold: A training session was

In [11]:
IVEC_CTX = {q["id"]: vector_search(q["q"]) for q in KQ}
IVEC = llm.run("indep_vector", {q["id"]: LLM.body([{"role": "system", "content": "You are the Aura IT helpdesk assistant. " + RULES},
                                                  {"role": "user", "content": f"Question: {q['q']}\n\nContext:\n" +
                                                   "\n\n".join(c['text'] for c, _ in IVEC_CTX[q['id']])}], max_tokens=800) for q in KQ})
istate = {q["id"]: {"msgs": [{"role": "system", "content": AGENT_SYS}, {"role": "user", "content": q["q"]}],
                    "answer": None, "steps": 0} for q in KQ}
for step in range(10):
    active = [k for k, s in istate.items() if s["answer"] is None]
    if not active:
        break
    res = llm.run(f"indep_agent_step{step + 1}", {k: LLM.body(istate[k]["msgs"], max_tokens=800, tools=TOOLS) for k in active})
    for k in active:
        r, s = res[k], istate[k]
        s["steps"] += 1
        if not r["tool_calls"]:
            s["answer"] = r["text"]; continue
        s["msgs"].append({"role": "assistant", "content": r["text"] or None,
                          "tool_calls": [{"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["arguments"]}}
                                         for t in r["tool_calls"]]})
        for t in r["tool_calls"]:
            try:
                out = IMPL[t["name"]](**json.loads(t["arguments"]))
            except Exception as e:
                out = json.dumps({"error": f"{type(e).__name__}: {e}"})
            s["msgs"].append({"role": "tool", "tool_call_id": t["id"], "content": out})
    print(f"step {step + 1}: {len(active)} agents active")
for s in istate.values():
    if s["answer"] is None:
        s["answer"] = "(no answer within step budget)"
print(f"Average tool steps per question: {np.mean([s['steps'] for s in istate.values()]):.1f}")

[indep_vector] 115 new requests -> test (0 cached)


[indep_agent_step1] 115 new requests -> test (0 cached)


step 1: 115 agents active
[indep_agent_step2] 115 new requests -> test (0 cached)


step 2: 115 agents active
[indep_agent_step3] 106 new requests -> test (0 cached)


step 3: 106 agents active
[indep_agent_step4] 60 new requests -> test (0 cached)


...

step 6: 35 agents active
[indep_agent_step7] 25 new requests -> test (0 cached)


step 7: 25 agents active
[indep_agent_step8] 22 new requests -> test (0 cached)


step 8: 22 agents active
[indep_agent_step9] 18 new requests -> test (0 cached)


step 9: 18 agents active
[indep_agent_step10] 13 new requests -> test (0 cached)


step 10: 13 agents active
Average tool steps per question: 4.8


In [12]:
class Grade(BaseModel):
    correct: bool
GRADE_SYS = ("You grade helpdesk answers strictly against a gold answer. correct=true only if the candidate gives the "
             "gold answer's facts (all numbers, names and items; for a problem report, the same fix as its main "
             "recommendation) and nothing that contradicts them. Extra correct detail is fine.")
ig = llm.run("indep_judge", {f"{m}|{q['id']}": LLM.body([{"role": "system", "content": GRADE_SYS}, {"role": "user", "content":
    f"Question: {q['q']}\nGold answer: {q['gold']}\n\nCandidate answer:\n{ans}"}], max_tokens=20, schema=Grade)
    for q in KQ for m, ans in (("vector", IVEC[q["id"]]["text"]), ("graph", istate[q["id"]]["answer"]))}, parse=Grade)
IOK = {(q["id"], m): bool((ig[f"{m}|{q['id']}"]["parsed"] or {}).get("correct")) for q in KQ for m in ("vector", "graph")}
IRES = {}
print(f"{'family':<18}{'n':>4}{'vector [95% CI]':>22}{'graph agent [95% CI]':>24}{'graph - vector [95% CI]':>28}")
for f in FAMS + ["ALL"]:
    qs = [q for q in KQ if f == "ALL" or q["family"] == f]
    row = {}
    for m in ("vector", "graph"):
        k = sum(IOK[(q["id"], m)] for q in qs); lo, hi = wilson(k, len(qs)); row[m] = (k / len(qs), lo, hi)
    d = paired_boot([IOK[(q["id"], "vector")] for q in qs], [IOK[(q["id"], "graph")] for q in qs])
    IRES[f] = (len(qs), row, d)
    fmt = lambda t: f"{t[0]:.0%} [{t[1]:.0%}-{t[2]:.0%}]"
    print(f"{f:<18}{len(qs):>4}{fmt(row['vector']):>22}{fmt(row['graph']):>24}{f'{d[0]:+.0%} [{d[1]:+.0%}, {d[2]:+.0%}]':>28}")

[indep_judge] 230 new requests -> test (0 cached)


family               n       vector [95% CI]    graph agent [95% CI]     graph - vector [95% CI]
A. lookup           29       100% [88%-100%]           86% [69%-95%]            -14% [-28%, -3%]
B. verified fix     28         50% [33%-67%]           46% [30%-64%]            -4% [-25%, +18%]
C. relational       58         67% [54%-78%]           60% [47%-72%]             -7% [-21%, +9%]
ALL                115         71% [62%-79%]           63% [54%-72%]             -8% [-17%, +2%]


In [13]:
# Could the kept multi-record questions have come from the 9 templates of cell 5? (LLM classification)
TEMPLATES = ["How many tickets are assigned to <user>?", "Which tickets assigned to <user> are not yet closed?",
             "How many of the tickets submitted by <user> are still not closed?", "How many comments has <user> posted?",
             "How many tickets are in category <c>, and how many of them are closed?", "How many tickets have priority <p>?",
             "Who wrote the accepted solution for ticket <t>, and what is their role?",
             "Which KB articles are referenced in the comments of ticket <t>?",
             "What is the status of ticket <t>, and who is it assigned to?",
             "Which users with role <r> have written at least one accepted solution?"]
class Tmpl(BaseModel):
    expressible: bool
KC = [q for q in KQ if q["family"] == "C. relational"]
tc = llm.run("indep_template_cls", {q["id"]: LLM.body([{"role": "user", "content": "Templates:\n" + "\n".join(f"- {t}" for t in TEMPLATES) +
    f"\n\nQuestion: {q['q']}\nexpressible=true only if the question asks the same thing as one of the templates with "
    "its slot filled (wording may differ)."}], max_tokens=20, schema=Tmpl) for q in KC}, parse=Tmpl)
expr = [q for q in KC if (tc[q["id"]]["parsed"] or {}).get("expressible")]
print(f"Kept multi-record questions expressible by an existing template (LLM classification): {len(expr)}/{len(KC)}")
for sub, name in ((expr, "template-like"), ([q for q in KC if q not in expr], "not template-like")):
    if sub:
        a = [IOK[(q["id"], "vector")] for q in sub]; b = [IOK[(q["id"], "graph")] for q in sub]
        d = paired_boot(a, b)
        print(f"  {name:<18} n={len(sub):<3} vector {np.mean(a):.0%} | graph {np.mean(b):.0%} | "
              f"graph - vector {d[0]:+.0%} [{d[1]:+.0%}, {d[2]:+.0%}]")

[indep_template_cls] 58 new requests -> test (0 cached)


Kept multi-record questions expressible by an existing template (LLM classification): 0/58
  not template-like  n=58  vector 67% | graph 60% | graph - vector -7% [-21%, +9%]


In [14]:
LEDGER_INDEP = llm.print_ledger([s for s in llm.ledger() if s.startswith("indep_")])

stage                           calls     input tok   output tok      USD  via Batch
indep_agent_step1                 115        57,435        4,507     0.01         0%
indep_agent_step10                 13        56,698          804     0.01         0%
indep_agent_step2                 115       122,499        3,919     0.01         0%
indep_agent_step3                 106       149,230        4,126     0.02         0%
indep_agent_step4                  60       115,670        2,393     0.01         0%
indep_agent_step5                  43        97,791        1,651     0.01         0%
indep_agent_step6                  35        90,719        1,678     0.01         0%
indep_agent_step7                  25        77,835        1,042     0.01         0%
indep_agent_step8                  22        76,157          884     0.01         0%
indep_agent_step9                  18        69,584          799     0.01         0%
indep_equiv                       119        21,751        1,309 